In [ ]:
# import
import torch
import random
import torch.nn as nn
import torch.nn.functional as F

import warnings

warnings.filterwarnings("ignore")

seed = 333

# Hyperparameter

In [ ]:
hp = {
    "t": 8,
    "c": 32,
    "nhead": 2,
    "nblock": 4,
    "dropout": 0.1,  # percentage to be dropped in dropout layer
    "hs": None,  # ignore this
}

assert hp["c"] % hp["nhead"] == 0, "c must be divisible by nhead"

hp["hs"] = hp["c"] // hp["nhead"]

# Data

## Import Data

In [ ]:
# import data
with open("../data/input.txt", "r", newline="\n") as file:
    data_file = file.read()

data_file[:100]

## Data Class

In [ ]:
class Data:
    def __init__(
        self,
        _data: str,
        seed: int = seed,
        train_batch_size: int = None,
        train_block_size: int = None,
        test_batch_size: int = None,
        test_block_size: int = None,
    ) -> None:
        # initialisation
        self.seed = seed
        self.data = _data  # raw data without process
        self.len_data = len(_data)  # how many character in raw data
        self.char_list = sorted(list(set(_data)))  # list of possible character
        self.len_char = len(self.char_list)  # len of possible character

        # encoder and decoder list
        self.enc_dict = {key: value for value, key in enumerate(self.char_list)}  # convert to number
        self.dec_dict = {key: value for key, value in enumerate(self.char_list)}  # convert to string

        # encoded data
        self.data_enc = self.encoder(self.data)  # raw data without process

        # split to train and test set with ration 8:2
        split_at_index = self.len_data // 8
        self.train = self.data_enc[:split_at_index]  # full train set
        self.test = self.data_enc[split_at_index:]  # full test set

        # create x,y for train and test set
        if not None in [train_batch_size, test_batch_size, train_block_size, test_block_size]:
            self.x_train, self.y_train = self.sample_batch(
                batch_size=train_batch_size, block_size=train_block_size, input=self.train
            )
            self.x_test, self.y_test = self.sample_batch(
                batch_size=test_batch_size, block_size=test_block_size, input=self.test
            )

    def encoder(self, input: str) -> torch.Tensor:
        """convert string to encoded index

        Args:
            input (str): string to be encoded

        Returns:
            torch.Tensor: (copy) tensor of encoded index
        """
        return torch.tensor([self.enc_dict[char] for char in input])

    def decoder(self, input: torch.Tensor) -> str:
        """covnert encoded index in torch.tensor into

        Args:
            input (torch.Tensor): tensor of encoded index

        Returns:
            str: (copy) string from encoded index in tensor(input)
        """
        return "".join([self.dec_dict[i.item()] for i in input])

    def sample_batch(self, batch_size: int, block_size: int, input: torch.Tensor):
        """generate sample batch

        Args:
            batch_size (int): how many batch generated
            block_size (int): maximum length per batch
            input (torch.Tensor): tensor of encoded data

        Returns:
            x,y: data_x,data_y
        """
        output_x = torch.empty((batch_size, block_size), dtype=torch.long)
        output_y = (
            torch.empty((batch_size, block_size), dtype=torch.long)
            if block_size != 1
            else torch.empty((batch_size), dtype=torch.long)
        )
        len_input = len(input)

        for batch_index in range(batch_size):
            offset = random.randint(0, len_input - block_size - 1)
            output_x[batch_index] = input[offset : offset + block_size]
            output_y[batch_index] = input[offset + 1 : offset + block_size + 1]
        return output_x, output_y

In [ ]:
d = Data(
    _data=data_file,
    seed=seed,
    train_batch_size=50000,
    test_batch_size=50000,
    train_block_size=hp["t"],
    test_block_size=hp["t"],
)

# Layer/Block

In [ ]:
class Head(nn.Module):
    def __init__(
        self,
        c: int,
        head_size: int,
    ) -> None:
        """
        - (b,t,c) -> (b,t,head_size)
        - headsize*nhead = c
        Args:
            c (int): input size (_,_,c)
            head_size (int): output size (_,_,hs)
        """
        super().__init__()
        self.c = c

        # layers
        self.query = nn.Linear(c, head_size)
        self.key = nn.Linear(c, head_size)
        self.value = nn.Linear(c, head_size)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        """forward prop

        Args:
            input (torch.Tensor): feed in embedded input data,shape(b,t,head_size)

        Returns:
            torch.Tensor: output tensor with shape # (b,t,head_size)
        """
        # get q,k,v
        q = self.query(input)  # shape: (b,t, head_size)
        k = self.key(input)  # shape: (b,t, head_size)
        v = self.value(input)  # shape: (b,t, head_size)

        # get attention
        attention_weight = (q @ k.transpose(-2, -1)) * (self.c**-0.5)  # shape: (b,t,t)
        self.register_buffer("tril", torch.ones_like(attention_weight).triu(diagonal=1).bool())
        attention_weight = attention_weight.masked_fill(self.tril, float("-inf"))  # shape: (b,t,t)
        attention_weight = F.softmax(attention_weight, dim=-1)  # shape: (b,t,t)

        # output
        out = attention_weight @ v  # shape: (b,t,head_size)

        return out

In [ ]:
class MultiHead(nn.Module):
    def __init__(
        self,
        c: int,
        nhead: int,
        hs: int,
    ) -> None:
        """multihead attention
        - (b,t,c) -> (b,t,c)
        Args:
            c (int): (_,_,c)
            nhead (int): nhead*hs = c
            hs (int): nhead*hs = c
        """
        super().__init__()
        self.heads = nn.ModuleList([Head(c, hs) for _ in range(nhead)])
        self.drop = nn.Dropout(hp["dropout"])
        self.linear = nn.Linear(c, c)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        """forward prop

        Args:
            input (torch.Tensor): input size (b,t,c)

        Returns:
            torch.Tensor: output size (b,t,c)
        """
        x = torch.cat([head(input) for head in self.heads], dim=-1)
        x = self.drop(self.linear(x))

        return x

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, c: int) -> None:
        """
        Args:
            c (int): input size (_,_,c)
        """
        super().__init__()

        self.sequential = nn.Sequential(nn.Linear(c, c * 4), nn.ReLU(), nn.Linear(c * 4, c), nn.Dropout(hp["dropout"]))

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        """forward prop

        Args:
            input (torch.Tensor): input shape (b,t,c)

        Returns:
            torch.Tensor: output shape (b,t,c)
        """
        return self.sequential(input)

In [ ]:
class Block(nn.Module):
    def __init__(
        self,
        c: int,
        nhead: int,
        hs: int,
    ) -> None:
        """
        Args:
            c (int, optional): (_,_,c)
            nhead (int, optional): nhead*hs = c
            hs (int, optional): nhead*hs = c
        """
        super().__init__()

        self.heads1 = MultiHead(c, nhead, hs)
        self.lnorm1 = nn.LayerNorm(c)
        self.heads2 = MultiHead(c, nhead, hs)
        self.lnorm2 = nn.LayerNorm(c)
        self.feedforward = FeedForward(c)
        self.lnorm3 = nn.LayerNorm(c)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        """(b,t,c) -> (b,t,c)

        Args:
            input (torch.Tensor): (b,t,c)

        Returns:
            torch.Tensor: (b,t,c)
        """
        x = input

        x = x + self.heads1(x)
        x = self.lnorm1(x)

        x = x + self.heads2(x)
        x = self.lnorm2(x)

        x = x + self.feedforward(x)
        x = self.lnorm3(x)

        return x

# Main Model

In [ ]:
class Model(nn.Module):
    def __init__(
        self,
        t: int = hp["t"],
        c: int = hp["c"],
        nhead: int = hp["nhead"],
        hs: int = hp["hs"],
        nblock: int = hp["nblock"],
    ):
        # initialisation
        super().__init__()

        # embedding
        self.emb = nn.Embedding(d.len_char, c)  # out = (b,t,c)
        self.pos_emb = nn.Embedding(t, c)  # out = (b,t,c)

        # block
        self.blocks = nn.Sequential(*[Block(c, nhead, hs) for _ in range(nblock)])  # out = (b,t,c)

        # d.len_char linear transform
        self.linear = nn.Linear(c, d.len_char)

        # flatten
        self.flatten = nn.Flatten(0, 1)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        """forward prop

        Args:
            input (torch.Tensor): input tensor with shape (batch_size,block_size)

        Returns:
            torch.Tensor: logit tensor with shape (batch_size,len_char=65)
        """
        # embedding
        x = self.emb(input) + self.pos_emb(torch.arange(input.shape[1]))  # shape = (b,t,c)

        # layers
        x: torch.Tensor = self.blocks(x)  # shape = (b,t,c)
        logits: torch.Tensor = self.linear(x)  # shape = (b,t,len_char)

        if logits.ndim == 3:
            logits: torch.Tensor = self.flatten(logits)  # shape = (b*t,len_char)

        return logits

    def loss(self, logits: torch.Tensor, expected: torch.Tensor):
        """calculate loss

        Args:
            logits (torch.Tensor): logit tensor with size (batch_size,len_char)
            expected (torch.Tensor): expected class tensor with shape (batch size,categorical number within len_char)

        Returns:
            loss_value in tensor
        """
        return F.cross_entropy(logits, expected.view(-1))

    @torch.no_grad()
    def loss_test(self):
        """calculate loss of test set

        Returns:
            loss value of test set
        """
        return self.loss(self(d.x_test), d.y_test).item()

    def fit(
        self,
        x: torch.Tensor,
        y: torch.Tensor,
        lr: float = 1e-3,
        lr_step_size: int = 1000,
        lr_gamma: float = 0.9,
        batch_size: int = 100,
        iteration: int = 10000,
        print_iteration: int = 1000,
    ) -> None:
        # optimizer
        optimiser = torch.optim.AdamW(self.parameters(), lr=lr)
        lr_decay = torch.optim.lr_scheduler.StepLR(optimiser, step_size=lr_step_size, gamma=lr_gamma)

        for i in range(iteration):
            # batch x,y
            batch_index = torch.randint(low=0, high=y.shape[0], size=(batch_size,))
            batch_x = x[batch_index]
            batch_y = y[batch_index]
            batch_y = batch_y.view(-1)

            loss = self.loss(logits=self(batch_x), expected=batch_y)
            optimiser.zero_grad(set_to_none=True)
            loss.backward()
            optimiser.step()
            lr_decay.step()

            if ((i + 1) % print_iteration == 0) or ((i + 1) == iteration) or (i == 0):
                if i == 0:
                    print("                      Loss_Train : Loss test")
                with torch.no_grad():
                    print(f"{i+1}th Iteration = {self.loss(self.forward(x), y):.4f} : {self.loss_test():.4f}")

    @torch.no_grad()
    def sampling(self, idx, number_of_word: int):
        """generate sample

        Args:
            idx (torch.tensor): shape = (t)
            number_of_word (int): number of word to be generated
        """
        self.eval()  # eva mode

        out = idx
        b, t = idx.shape

        for _ in range(number_of_word):
            # get last char
            last_char_enc = out[:, -hp["t"] :]

            # forward
            logits = self(last_char_enc)[-1]
            softmax = F.softmax(logits)

            # get next char
            next_char_enc = torch.multinomial(softmax, 1)

            # append
            out = torch.cat((out, next_char_enc.view(b, 1)), dim=-1)
        print(d.decoder(out.view(-1)))

        self.train()  # train mode
    
    def summary(self) -> None:
        """print model summary"""
        npara = 0
        for p in self.parameters():
            npara += p.numel()
        print(f"Number of parameter = {npara}\n")

In [ ]:
m = Model()
m.summary()
m.fit(
    x=d.x_train,
    y=d.y_train,
    lr=1e-3,
    lr_step_size=2000,
    lr_gamma=1,  # set to 1 to disable
    batch_size=32,
    iteration=1,
    print_iteration=1,
)

In [ ]:
# m.sampling(idx=torch.randint(0, d.len_char, size=(1, 1)), number_of_word=1000)